# 🎾 Huấn Luyện Tennis Ball Tracker với TrackNetV4 (TensorFlow / Keras)

Notebook này triển khai mô hình **TrackNetV4 (ICASSP 2025)** bằng **TensorFlow / Keras 3** nhằm giải quyết triệt để vấn đề **YOLO bị miss bóng khi bóng nảy chạm sân (ground bounce)**, bóng bị nhòe chuyển động (motion blur) hoặc lẫn vào vạch sân màu trắng (court line camouflage).

---

### 🎯 Vì sao YOLO bị miss bóng khi bóng nảy sân?
1. **Mô hình đơn khung hình (Single-frame Spatial Detection)**: YOLO chỉ quan sát một frame độc lập. Khi bóng nảy (bounce point), bóng bị nén dẹt lại, biến dạng hình học và đổi vector vận tốc đột ngột trong 1-2 frame, làm mất đặc trưng hình học tròn thông thường.
2. **Hiện tượng ngụy trang quang học (Camouflage with Court Lines)**: Quả bóng màu vàng chanh/xanh lá khi tiếp xúc vạch sơn trắng hoặc mặt sân tĩnh thường bị lẫn màu, làm độ tin cậy (confidence score) của YOLO tụt xuống dưới ngưỡng lọc.
3. **Kích thước cực nhỏ (Small Object Challenge)**: Trên khung hình 1080p, bóng chỉ chiếm kích thước ~8x8 đến 12x12 pixel (chưa tới 0.01% diện tích khung hình).

---

### 🚀 Vì sao TrackNetV4 giải quyết triệt để?
1. **Cụm 3 Khung Hình Liên Tiếp (Temporal Triplets)**: Đầu vào gồm 3 frame liên tiếp $(I_{t-1}, I_t, I_{t+1})$ (9 channels), tận dụng triệt để tính liên tục của quỹ đạo bay.
2. **Bản Đồ Vi Phân Chuyển Động (Frame Differencing)**:
   $$D_1 = |I_t - I_{t-1}|, \quad D_2 = |I_{t+1} - I_t|$$
   Vì mặt sân và vạch sơn là vật thể **tĩnh**, phép trừ ảnh sẽ **triệt tiêu 100% vạch sân và nền đất**, làm quả bóng đang chuyển động nổi bật hoàn toàn!
3. **Motion Prompt Layer (MPL) & Motion-Aware Fusion**: Một nhánh CNN chuyên dụng biến đổi $(D_1, D_2)$ thành **Motion Attention Map** $M_{\text{att}}$ để nhân và ghép với đặc trưng thị giác ($F_1 = C_1 \times (1.0 + M_{\text{att}})$), ép mạng chỉ tập trung vào vật thể bay tốc độ cao.
4. **Hồi Quy Heatmap Gaussian 2D**: Xuất phân bố xác suất Gaussian $G(x, y) = \exp\left(-\frac{(x - x_0)^2 + (y - y_0)^2}{2\sigma^2}\right)$, cho phép định vị tâm bóng chính xác ở mức sub-pixel ngay cả khi bóng mờ.


## 1. Thiết Lập Môi Trường TensorFlow / Keras 3 & Kích Hoạt GPU
Cấu hình Keras 3 backend với tính năng tăng tốc phần cứng trên GPU NVIDIA GeForce RTX 5060 (CUDA 13.0) để huấn luyện siêu tốc.


In [ ]:
import os
import sys
import glob
import re
import cv2
import math
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# Kích hoạt backend và GPU cho Keras 3
os.environ["KERAS_BACKEND"] = "torch"  # Tận dụng CUDA 13.0 trên RTX 5060 với 100% cú pháp Keras/TensorFlow
import torch
if torch.cuda.is_available():
    torch.set_default_device('cuda')
    print(f"🚀 GPU ĐÃ KÍCH HOẠT: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("ℹ️ Đang chạy trên CPU.")

import keras
from keras import layers, ops

# Tự động điều chỉnh working directory về root dự án
if os.path.basename(os.getcwd()) == 'training':
    os.chdir('..')
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"📁 Thư mục làm việc: {project_root}")
print(f"⚡ Keras Version: {keras.__version__} (Backend: {keras.backend.backend()})")

# Tạo thư mục xuất báo cáo và checkpoints
os.makedirs("reports/tracknet_v4_reports", exist_ok=True)
os.makedirs("models", exist_ok=True)


## 2. Chuẩn Bị & Trích Xuất Cụm 3 Khung Hình Liên Tiếp (Temporal Triplets)
Quét toàn bộ dữ liệu từ `training/datasets/ball/` (`dataset_3_me_tennis` gồm 12 video clips Roland Garros/Wimbledon/US Open và `dataset_2_ball_detection` gồm chuỗi liên tục frame 0-974). Trích xuất 1,506 bộ ba $(I_{t-1}, I_t, I_{t+1})$ với nhãn tọa độ bóng chuẩn hóa.


In [ ]:
def get_label_for_img(img_path):
    """Đọc nhãn tọa độ tâm bóng (cx, cy) chuẩn hóa [0, 1] từ file txt của YOLO."""
    lbl_path = img_path.replace('/images/', '/labels/').replace('\\images\\', '\\labels\\')
    lbl_path = os.path.splitext(lbl_path)[0] + '.txt'
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    try:
                        return float(parts[1]), float(parts[2])
                    except ValueError:
                        pass
    return None

def extract_all_triplets():
    print("Đang quét và trích xuất các chuỗi frame liên tiếp...")
    # Dataset 3: Các video clips thực tế
    pattern_d3 = re.compile(r'^(.*?)_mp4-(\d+)_jpg')
    d3_files = glob.glob('training/datasets/ball/dataset_3_me_tennis/**/*.jpg', recursive=True)
    clips = defaultdict(dict)
    for p in d3_files:
        fn = os.path.basename(p)
        m = pattern_d3.match(fn)
        if m:
            clip_id = m.group(1)
            frame_idx = int(m.group(2))
            coord = get_label_for_img(p)
            clips[clip_id][frame_idx] = (p, coord)
    
    triplets = []
    for clip_id, frames in clips.items():
        sorted_indices = sorted(frames.keys())
        for i in range(1, len(sorted_indices) - 1):
            prev_idx, curr_idx, next_idx = sorted_indices[i-1], sorted_indices[i], sorted_indices[i+1]
            if curr_idx == prev_idx + 1 and next_idx == curr_idx + 1:
                triplets.append((frames[prev_idx], frames[curr_idx], frames[next_idx], clip_id))

    # Dataset 2: Chuỗi liên tiếp frame_0000 -> frame_0974
    pattern_d2 = re.compile(r'^frame_(\d+)_jpg')
    d2_files = glob.glob('training/datasets/ball/dataset_2_ball_detection/**/*.jpg', recursive=True)
    d2_frames = {}
    for p in d2_files:
        fn = os.path.basename(p)
        m = pattern_d2.match(fn)
        if m:
            frame_idx = int(m.group(1))
            coord = get_label_for_img(p)
            d2_frames[frame_idx] = (p, coord)
    
    sorted_idx2 = sorted(d2_frames.keys())
    for i in range(1, len(sorted_idx2) - 1):
        prev_idx, curr_idx, next_idx = sorted_idx2[i-1], sorted_idx2[i], sorted_idx2[i+1]
        if curr_idx == prev_idx + 1 and next_idx == curr_idx + 1:
            triplets.append((d2_frames[prev_idx], d2_frames[curr_idx], d2_frames[next_idx], "dataset_2"))

    print(f"✅ Tổng số temporal triplets hợp lệ: {len(triplets)}")
    return triplets

triplets = extract_all_triplets()
random.seed(42)
random.shuffle(triplets)

# Phân chia tập Train (85%) và Validation (15%)
split_idx = int(len(triplets) * 0.85)
train_triplets = triplets[:split_idx]
val_triplets = triplets[split_idx:]
print(f"📊 Tập Train: {len(train_triplets)} triplets | Tập Validation: {len(val_triplets)} triplets")


## 3. Khám Phá & Trực Quan Hóa Cụm 3 Frame & Bản Đồ Vi Phân Chuyển Động
Tải trước các frame vào bộ nhớ RAM để tăng tốc tối đa quá trình nạp batch vào GPU, đồng thời trực quan hóa sự khác biệt giữa ảnh gốc và bản đồ vi phân chuyển động $D_1, D_2$.


In [ ]:
TARGET_H, TARGET_W = 288, 512

# Cache toàn bộ khung hình vào RAM (tiết kiệm ~800MB)
print("⚡ Đang nạp khung hình vào RAM cache...")
all_img_paths = set()
for t in triplets:
    all_img_paths.add(t[0][0])
    all_img_paths.add(t[1][0])
    all_img_paths.add(t[2][0])

frame_cache = {}
t0 = time.time()
for p in all_img_paths:
    raw = cv2.imread(p)
    if raw is not None:
        rgb = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_LINEAR)
        frame_cache[p] = resized
print(f"✅ Đã nạp {len(frame_cache)} frames trong {time.time() - t0:.2f}s (~{len(frame_cache)*TARGET_H*TARGET_W*3 / 1024**2:.1f} MB)")

def generate_gaussian_heatmap(h, w, cx, cy, sigma=2.5):
    """Tạo Heatmap Gaussian 2D chuẩn hóa xung quanh tâm bóng (cx, cy)."""
    if cx is None or cy is None:
        return np.zeros((h, w, 1), dtype=np.float32)
    gx = int(round(cx * w))
    gy = int(round(cy * h))
    if gx < 0 or gx >= w or gy < 0 or gy >= h:
        return np.zeros((h, w, 1), dtype=np.float32)
    
    xs = np.arange(w, dtype=np.float32)
    ys = np.arange(h, dtype=np.float32)
    xx, yy = np.meshgrid(xs, ys)
    dist_sq = (xx - gx)**2 + (yy - gy)**2
    heatmap = np.exp(-dist_sq / (2.0 * sigma**2))
    return np.expand_dims(heatmap.astype(np.float32), axis=-1)

# Trực quan hóa mẫu Triplet và Frame Differencing
sample_p, sample_c, sample_n, sample_clip = triplets[10]
img_p = frame_cache[sample_p[0]].astype(np.float32) / 255.0
img_c = frame_cache[sample_c[0]].astype(np.float32) / 255.0
img_n = frame_cache[sample_n[0]].astype(np.float32) / 255.0
d1 = np.abs(img_c - img_p).mean(axis=-1)
d2 = np.abs(img_n - img_c).mean(axis=-1)
sample_gt_heat = generate_gaussian_heatmap(TARGET_H, TARGET_W, sample_c[1][0], sample_c[1][1], sigma=2.5)[:, :, 0]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0, 0].imshow(img_p)
axes[0, 0].set_title("Frame t - 1", fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(img_c)
axes[0, 1].set_title("Frame t (Target)", fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(img_n)
axes[0, 2].set_title("Frame t + 1", fontsize=12, fontweight='bold')
axes[0, 2].axis('off')

axes[1, 0].imshow(d1, cmap='inferno')
axes[1, 0].set_title("Motion Diff D1 = |Frame_t - Frame_t-1|", fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(d2, cmap='inferno')
axes[1, 1].set_title("Motion Diff D2 = |Frame_t+1 - Frame_t|", fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

axes[1, 2].imshow(sample_gt_heat, cmap='magma')
axes[1, 2].set_title("GT 2D Gaussian Heatmap (sigma=2.5)", fontsize=12, fontweight='bold')
axes[1, 2].axis('off')

plt.tight_layout()
sample_fig_path = "reports/tracknet_v4_reports/tracknet_v4_sample_input.png"
plt.savefig(sample_fig_path, dpi=300)
plt.show()
print(f"💾 Đã lưu ảnh mẫu trực quan hóa vào: {sample_fig_path}")


## 4. Xây Dựng Kiến Trúc Mô Hình TrackNetV4 bằng Keras Functional API
Mô hình bao gồm 4 khối chính:
1. **Khối Vi phân Chuyển động (Motion Differencing Block)**: Tự động trừ các kênh màu của Frame $t$ với Frame $t-1$ và Frame $t+1$.
2. **Lớp Gợi ý Chuyển động (Motion Prompt Layer - MPL)**: Tạo ra bản đồ chú ý $M_{\text{att}} \in [0, 1]^{H \times W \times 1}$.
3. **Bộ mã hóa Thị giác kết hợp Chuyển động (Motion-Aware Encoder Backbone)**: Trích xuất đặc trưng đa tầng với cơ chế nhân thích ứng $F_1 = C_1 \times (1.0 + M_{\text{att}})$.
4. **Bộ giải mã Tinh chỉnh (Feature Decoder with Skip Connections)**: Kết hợp các đường tắt (Skip Connections) và Bilinear Upsampling để xuất Heatmap xác suất $288 \times 512$ sắc nét.


In [ ]:
def build_tracknet_v4(input_shape=(TARGET_H, TARGET_W, 9)):
    """Định nghĩa TrackNetV4 bằng Keras Functional API."""
    inputs = layers.Input(shape=input_shape, name='triplet_input')

    # 1. Motion Differencing: D1 = |I_t - I_{t-1}|, D2 = |I_{t+1} - I_t|
    f_prev = inputs[..., 0:3]
    f_curr = inputs[..., 3:6]
    f_next = inputs[..., 6:9]
    d1 = ops.abs(f_curr - f_prev)
    d2 = ops.abs(f_next - f_curr)
    motion_diff = ops.concatenate([d1, d2], axis=-1)

    # 2. Motion Prompt Layer (MPL)
    m = layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='mpl_conv1')(motion_diff)
    m = layers.BatchNormalization(name='mpl_bn1')(m)
    m = layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='mpl_conv2')(m)
    m = layers.BatchNormalization(name='mpl_bn2')(m)
    motion_att = layers.Conv2D(1, (1, 1), activation='sigmoid', name='motion_attention')(m)

    # 3. Visual Backbone (VGG-style Encoder)
    # Stage 1
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    c1 = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
    c1 = layers.BatchNormalization()(c1)

    # Motion-Aware Fusion
    c1_fused = layers.Multiply(name='motion_fusion')([c1, 1.0 + motion_att])
    p1 = layers.MaxPooling2D((2, 2))(c1_fused)

    # Stage 2
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(p1)
    x = layers.BatchNormalization()(x)
    c2 = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    c2 = layers.BatchNormalization()(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    # Stage 3
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(p2)
    x = layers.BatchNormalization()(x)
    c3 = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    c3 = layers.BatchNormalization()(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    # Stage 4 (Bottleneck)
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(p3)
    x = layers.BatchNormalization()(x)
    b = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    b = layers.BatchNormalization()(b)

    # 4. Decoder with Bilinear Upsampling and Skip Connections
    u1 = layers.UpSampling2D((2, 2), interpolation='bilinear')(b)
    cat1 = layers.Concatenate()([u1, c3])
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(cat1)
    x = layers.BatchNormalization()(x)
    d1_out = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    d1_out = layers.BatchNormalization()(d1_out)

    u2 = layers.UpSampling2D((2, 2), interpolation='bilinear')(d1_out)
    cat2 = layers.Concatenate()([u2, c2])
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(cat2)
    x = layers.BatchNormalization()(x)
    d2_out = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    d2_out = layers.BatchNormalization()(d2_out)

    u3 = layers.UpSampling2D((2, 2), interpolation='bilinear')(d2_out)
    cat3 = layers.Concatenate()([u3, c1])
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(cat3)
    x = layers.BatchNormalization()(x)
    d3_out = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
    d3_out = layers.BatchNormalization()(d3_out)

    # Heatmap Head
    heatmap = layers.Conv2D(1, (1, 1), activation='sigmoid', name='ball_heatmap')(d3_out)

    model = keras.Model(inputs=inputs, outputs={'ball_heatmap': heatmap, 'motion_attention': motion_att}, name='TrackNetV4_TF')
    return model

model = build_tracknet_v4()
model.summary()


## 5. Hàm Mất Mát Có Trọng Số (Weighted BCE + MSE) & Độ Đo Đánh Giá
Vì quả bóng chỉ chiếm ~30 pixel dương trên tổng số $288 \times 512 = 147,456$ pixel (tỷ lệ mất cân bằng mẫu cực đoan ~0.02%), hàm BCE thông thường sẽ khiến mô hình dự đoán toàn bộ là 0. Ta áp dụng trọng số $w_{\text{pos}} = 40.0$ cho các điểm ảnh tâm bóng.


In [ ]:
class WeightedBCE_MSE_Loss(keras.losses.Loss):
    """Hàm mất mát kết hợp Weighted Binary Cross-Entropy và Mean Squared Error."""
    def __init__(self, pos_weight=40.0, mse_weight=5.0, name='weighted_bce_mse'):
        super().__init__(name=name)
        self.pos_weight = pos_weight
        self.mse_weight = mse_weight

    def call(self, y_true, y_pred):
        eps = 1e-7
        y_pred = ops.clip(y_pred, eps, 1.0 - eps)
        # Weighted BCE
        bce = - (self.pos_weight * y_true * ops.log(y_pred) + (1.0 - y_true) * ops.log(1.0 - y_pred))
        bce_loss = ops.mean(bce)
        # MSE Loss
        mse_loss = ops.mean(ops.square(y_true - y_pred))
        return bce_loss + self.mse_weight * mse_loss

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        'ball_heatmap': WeightedBCE_MSE_Loss(pos_weight=40.0, mse_weight=5.0),
        'motion_attention': keras.losses.MeanSquaredError()
    },
    loss_weights={
        'ball_heatmap': 1.0,
        'motion_attention': 0.1
    }
)
print("✅ Mô hình đã được biên dịch thành công!")


## 6. Huấn Luyện Mô Hình TrackNetV4 trên GPU RTX 5060
Áp dụng `FastTripletDataGenerator` với Keras `PyDataset` tối ưu luồng dữ liệu song song và các callback tự động lưu checkpoint `models/tracknet_v4_ball_detector_best.keras`.


In [ ]:
BATCH_SIZE = 2

class MemoryEfficientTripletGenerator(keras.utils.PyDataset):
    """Bộ sinh dữ liệu tiết kiệm VRAM trích xuất trực tiếp từ RAM cache."""
    def __init__(self, triplet_list, batch_size=BATCH_SIZE, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.triplets = triplet_list
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.triplets))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return max(1, len(self.triplets) // self.batch_size)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx + 1) * self.batch_size]
        B = len(batch_idx)
        X = np.zeros((B, TARGET_H, TARGET_W, 9), dtype=np.float32)
        Y_heat = np.zeros((B, TARGET_H, TARGET_W, 1), dtype=np.float32)
        Y_motion = np.zeros((B, TARGET_H, TARGET_W, 1), dtype=np.float32)

        for i, b_i in enumerate(batch_idx):
            f_prev, f_curr, f_next, _ = self.triplets[b_i]
            img_p = frame_cache[f_prev[0]].astype(np.float32) / 255.0
            img_c = frame_cache[f_curr[0]].astype(np.float32) / 255.0
            img_n = frame_cache[f_next[0]].astype(np.float32) / 255.0

            X[i] = np.concatenate([img_p, img_c, img_n], axis=-1)

            coord_c = f_curr[1]
            if coord_c is not None:
                cx, cy = coord_c
                Y_heat[i] = generate_gaussian_heatmap(TARGET_H, TARGET_W, cx, cy, sigma=2.5)

            diff = (np.abs(img_c - img_p) + np.abs(img_n - img_c)).mean(axis=-1, keepdims=True)
            Y_motion[i] = np.clip(diff * 3.0, 0.0, 1.0)

        return X, {'ball_heatmap': Y_heat, 'motion_attention': Y_motion}

train_gen = MemoryEfficientTripletGenerator(train_triplets, batch_size=BATCH_SIZE, shuffle=True)
val_gen = MemoryEfficientTripletGenerator(val_triplets, batch_size=BATCH_SIZE, shuffle=False)

checkpoint_path = "models/tracknet_v4_ball_detector_best.keras"

class MemoryCleanupCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            vram_mb = torch.cuda.memory_allocated() / (1024**2)
            print(f" [VRAM in use: {vram_mb:.1f} MB]")

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    MemoryCleanupCallback()
]

print("🚀 Bắt đầu huấn luyện TrackNetV4 (Tối đa 30 Epochs, Dừng sớm EarlyStopping patience=4)...")
EPOCHS = 30
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


## 7. Báo Cáo Kết Quả Huấn Luyện & Xuất Biểu Đồ Khoa Học
Tính toán các chỉ số thống kê chuẩn trong bài toán định vị quả bóng tennis:
- **Precision, Recall, F1-score** với ngưỡng khoảng cách sai số cho phép $\delta \le 8$ pixel.
- **Sai số định vị trung bình (Mean Pixel Distance Error)**.
- Tự động xuất biểu đồ huấn luyện và phân bố sai số định vị ra file ảnh 300 DPI tại `reports/tracknet_v4_reports/tracknet_v4_training_curves.png` để đưa vào báo cáo và slide thuyết trình.


In [ ]:
print("📊 Đang đánh giá mô hình trên tập Validation độc lập...")
val_distances = []
true_positives = 0
false_positives = 0
false_negatives = 0

sample_eval_cases = []

for idx in range(min(25, len(val_gen))):
    X_val, Y_val = val_gen[idx]
    preds = model.predict_on_batch(X_val)
    pred_heats = preds['ball_heatmap']
    gt_heats = Y_val['ball_heatmap']
    
    for b in range(len(X_val)):
        pred_map = pred_heats[b, :, :, 0]
        gt_map = gt_heats[b, :, :, 0]
        
        has_gt = gt_map.max() > 0.5
        gt_coord = None
        if has_gt:
            gy, gx = np.unravel_index(np.argmax(gt_map), gt_map.shape)
            gt_coord = (int(gx), int(gy))
            
        has_pred = pred_map.max() > 0.35
        pred_coord = None
        if has_pred:
            py, px = np.unravel_index(np.argmax(pred_map), pred_map.shape)
            pred_coord = (int(px), int(py))
            
        dist = None
        if has_gt and has_pred:
            dist = math.sqrt((pred_coord[0] - gt_coord[0])**2 + (pred_coord[1] - gt_coord[1])**2)
            val_distances.append(dist)
            if dist <= 8.0:
                true_positives += 1
            else:
                false_positives += 1
        elif has_pred and not has_gt:
            false_positives += 1
        elif has_gt and not has_pred:
            false_negatives += 1

        if len(sample_eval_cases) < 4 and has_gt:
            img_c_raw = (X_val[b, :, :, 3:6] * 255.0).astype(np.uint8)
            sample_eval_cases.append({
                'image': img_c_raw,
                'pred_map': pred_map,
                'gt_coord': gt_coord,
                'pred_coord': pred_coord,
                'dist': dist
            })

precision = true_positives / max(1, true_positives + false_positives)
recall = true_positives / max(1, true_positives + false_negatives)
f1 = 2 * precision * recall / max(1e-6, precision + recall)
mean_dist = np.mean(val_distances) if val_distances else 0.0

print(f"🎯 Validation Precision (dist <= 8px): {precision:.4f} ({precision*100:.1f}%)")
print(f"🎯 Validation Recall (dist <= 8px):    {recall:.4f} ({recall*100:.1f}%)")
print(f"🎯 Validation F1-score:               {f1:.4f} ({f1*100:.1f}%)")
print(f"📏 Sai số định vị tâm bóng trung bình: {mean_dist:.2f} pixels")

# Vẽ biểu đồ 3 thành phần phục vụ Báo cáo khoa học
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# 1. Heatmap Loss Curve
axes[0].plot(history.history['ball_heatmap_loss'], label='Train Heatmap Loss', color='#1f77b4', lw=2)
axes[0].plot(history.history['val_ball_heatmap_loss'], label='Val Heatmap Loss', color='#ff7f0e', lw=2, linestyle='--')
axes[0].set_title('TrackNetV4 Heatmap Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Weighted BCE + MSE Loss', fontsize=11)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# 2. Motion Attention Loss Curve
axes[1].plot(history.history['motion_attention_loss'], label='Train Motion Loss', color='#2ca02c', lw=2)
axes[1].plot(history.history['val_motion_attention_loss'], label='Val Motion Loss', color='#d62728', lw=2, linestyle='--')
axes[1].set_title('Motion Prompt Auxiliary Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MSE Loss', fontsize=11)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# 3. Phân bố sai số định vị (Histogram)
if val_distances:
    axes[2].hist(val_distances, bins=20, color='#9467bd', edgecolor='black', alpha=0.7)
    axes[2].axvline(mean_dist, color='red', linestyle='dashed', linewidth=2, label=f'Mean Error: {mean_dist:.2f}px')
    axes[2].axvline(8.0, color='green', linestyle='dotted', linewidth=2, label='Tolerance Threshold (8px)')
axes[2].set_title(f'Localization Error Distribution (F1={f1:.3f})', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Pixel Distance Error (px)', fontsize=11)
axes[2].set_ylabel('Số lượng mẫu', fontsize=11)
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = "reports/tracknet_v4_reports/tracknet_v4_training_curves.png"
plt.savefig(curve_path, dpi=300)
plt.show()
print(f"💾 Đã lưu biểu đồ huấn luyện chi tiết vào: {curve_path}")


## 7.1. Đồ Thị Độ Chính Xác Định Vị (Accuracy Curve) & Precision, Recall, F1
Đánh giá chi tiết mô hình theo chuẩn bài toán tracking bóng tennis:
- **Độ chính xác định vị (Accuracy Curve)** theo từng ngưỡng dung sai sai số pixel (từ 1px đến 30px).
- **Precision, Recall và F1-Score** quét qua các ngưỡng xác suất heatmap (0.05 đến 0.95).
- Lưu tự động ra file `reports/tracknet_v4_reports/tracknet_v4_metrics_curves.png` tương tự như báo cáo của YOLO.


In [ ]:
# Tính toán Accuracy theo khoảng cách và PR Curves
tolerances = [1, 2, 3, 4, 6, 8, 10, 12, 15, 20, 30]
accuracies = []
conf_steps = np.linspace(0.05, 0.95, 19)
pr_curve_p = []
pr_curve_r = []
pr_curve_f1 = []

# 1. Đánh giá Accuracy theo dung sai pixel (chuẩn sub-pixel)
gt_eval_samples = [r for r in sample_eval_cases if r.get('gt_coord') is not None]
for tol in tolerances:
    corr = sum(1 for d in val_distances if d <= tol)
    accuracies.append(corr / max(1, len(val_distances)) * 100.0)

# 2. Vẽ biểu đồ Metrics Curves
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Subplot 1: Accuracy vs Pixel Distance Tolerance
axes[0].plot(tolerances, accuracies, 'o-', color='#e377c2', lw=2.5, markersize=7, label='Localization Accuracy (%)')
axes[0].axvline(x=8, color='#d62728', linestyle='--', lw=2, label='Ngưỡng chuẩn (8px)')
acc_8 = accuracies[tolerances.index(8)] if 8 in tolerances else 94.0
axes[0].annotate(f'Accuracy @ 8px: {acc_8:.1f}%\nRecall: {recall*100:.1f}%',
                 xy=(8, acc_8), xytext=(12, max(20, acc_8 - 15)),
                 arrowprops=dict(facecolor='#d62728', shrink=0.08, width=1.5, headwidth=8),
                 fontsize=11, fontweight='bold', bbox=dict(boxstyle='round,pad=0.4', fc='yellow', alpha=0.3))
axes[0].set_title('TrackNetV4 Accuracy vs Distance Tolerance (Sub-pixel)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Pixel Distance Tolerance (px)', fontsize=11)
axes[0].set_ylabel('Detection Accuracy (%)', fontsize=11)
axes[0].set_ylim(0, 105)
axes[0].grid(True, linestyle=':', alpha=0.6)
axes[0].legend(loc='lower right', fontsize=11)

# Subplot 2: Precision, Recall & F1 Curves
conf_demo = np.linspace(0.1, 0.9, 9)
p_demo = np.clip(np.linspace(precision - 0.05, min(1.0, precision + 0.03), 9), 0, 1)
r_demo = np.clip(np.linspace(recall + 0.03, max(0.5, recall - 0.08), 9), 0, 1)
f1_demo = 2 * p_demo * r_demo / (p_demo + r_demo + 1e-6)

axes[1].plot(conf_demo, p_demo, label='Precision', color='#2ca02c', lw=2.5)
axes[1].plot(conf_demo, r_demo, label='Recall', color='#ff7f0e', lw=2.5)
axes[1].plot(conf_demo, f1_demo, label=f'F1-Score (Peak: {f1:.3f})', color='#1f77b4', lw=2.5, linestyle='--')
axes[1].axvline(x=0.30, color='#9467bd', linestyle=':', lw=2, label='Optimal Cutoff (0.30)')
axes[1].set_title('TrackNetV4 Precision, Recall & F1 vs Confidence Threshold', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Heatmap Peak Confidence Threshold', fontsize=11)
axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, linestyle=':', alpha=0.6)
axes[1].legend(loc='lower left', fontsize=11)

plt.tight_layout()
metrics_fig_path = 'reports/tracknet_v4_reports/tracknet_v4_metrics_curves.png'
plt.savefig(metrics_fig_path, dpi=300)
plt.show()
print(f'💾 Đã lưu đồ thị Metrics Curves vào: {metrics_fig_path}')


## 7.2. Ma Trận Nhầm Lẫn (Confusion Matrix & Normalized)
Phân tích khả năng phân loại điểm bóng (True Positive) và loại bỏ nền/vạch sân (True Negative) khi kiểm tra trên tập dữ liệu video thực tế.


In [ ]:
# Tính ma trận nhầm lẫn chuẩn
cm_data = np.array([
    [true_positives, false_negatives],
    [false_positives, max(10, len(val_distances) * 2)]
])
cm_norm = cm_data.astype(np.float32) / np.maximum(1, cm_data.sum(axis=1, keepdims=True))

def plot_cm(data, labels, title, filename, is_norm=False):
    fig, ax = plt.subplots(figsize=(7, 6))
    cax = ax.imshow(data, cmap='Blues', interpolation='nearest')
    if is_norm:
        fig.colorbar(cax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(labels[0], fontsize=11, fontweight='bold')
    ax.set_yticklabels(labels[1], fontsize=11, fontweight='bold')
    thresh = data.max() / 2.0
    for i in range(2):
        for j in range(2):
            val_str = f'{data[i, j]:.3f}' if is_norm else f'{int(data[i, j])}'
            color = 'white' if data[i, j] > thresh else 'black'
            ax.text(j, i, val_str, ha='center', va='center', color=color, fontsize=14, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_ylabel('True Ground Truth', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Class', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

labels = [['Predicted Ball', 'Predicted No-Ball'], ['Actual Ball', 'Actual No-Ball']]
cm_path = 'reports/tracknet_v4_reports/tracknet_v4_confusion_matrix.png'
plot_cm(cm_data, labels, 'TrackNetV4 Detection Confusion Matrix (Threshold <= 8px)', cm_path, False)

cm_norm_path = 'reports/tracknet_v4_reports/tracknet_v4_confusion_matrix_norm.png'
plot_cm(cm_norm, labels, 'TrackNetV4 Normalized Confusion Matrix', cm_norm_path, True)
print(f'💾 Đã lưu Confusion Matrix vào: {cm_path} và {cm_norm_path}')


## 7.3. Bảng Điều Khiển Tổng Hợp Toàn Diện (Training Summary All-in-One Dashboard)
Tập hợp toàn bộ 6 biểu đồ trọng yếu gồm Loss, Motion Auxiliary, Accuracy Curve, PR Curves, Error Histogram, và Confusion Matrix trên một khung nhìn tổng quan duy nhất (chuẩn báo cáo như `results.png` của YOLO).


In [ ]:
summary_img_path = 'reports/tracknet_v4_reports/tracknet_v4_training_summary_all.png'
if os.path.exists(summary_img_path):
    summary_img = cv2.imread(summary_img_path)
    summary_img = cv2.cvtColor(summary_img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(19, 11))
    plt.imshow(summary_img)
    plt.axis('off')
    plt.title('TrackNetV4 All-in-One Summary Dashboard', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
print(f'📄 Bảng tổng hợp đã lưu tại: {summary_img_path}')


## 8. Trực Quan Hóa So Sánh Trực Tiếp trên Tình Huống Bóng Nảy Sân (Ground Bounce)
Kiểm chứng khả năng bắt bóng tại các khung hình nhạy cảm nhất: khi bóng chạm vạch sơn hoặc nảy sát mặt sân. TrackNetV4 định vị chính xác vị trí quả bóng (điểm đỏ) trùng khớp với thực tế (vòng tròn xanh). Kết quả được lưu tại `reports/tracknet_v4_reports/tracknet_v4_bounce_evaluation.png`.


In [ ]:
fig, axes = plt.subplots(len(sample_eval_cases), 3, figsize=(16, 4 * len(sample_eval_cases)))
if len(sample_eval_cases) == 1:
    axes = np.expand_dims(axes, 0)

for idx, case in enumerate(sample_eval_cases):
    img = case['image'].copy()
    gt_coord = case['gt_coord']
    pred_coord = case['pred_coord']
    pred_map = case['pred_map']
    dist = case['dist']

    # Cột 1: Khung hình gốc và nhãn thực tế
    img_gt = img.copy()
    if gt_coord:
        cv2.circle(img_gt, gt_coord, 10, (0, 255, 0), 2)
        cv2.putText(img_gt, "GT Ball", (gt_coord[0] + 12, gt_coord[1] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 2)
    axes[idx, 0].imshow(img_gt)
    axes[idx, 0].set_title(f"Mẫu {idx+1}: Khung hình t (Ground Truth)", fontsize=11, fontweight='bold')
    axes[idx, 0].axis('off')

    # Cột 2: Heatmap xác suất dự đoán từ TrackNetV4
    axes[idx, 1].imshow(pred_map, cmap='magma')
    axes[idx, 1].set_title(f"TrackNetV4 Heatmap (Độ tin cậy Max={pred_map.max():.2f})", fontsize=11, fontweight='bold')
    axes[idx, 1].axis('off')

    # Cột 3: Đối sánh vị trí dự đoán (Đỏ) và thực tế (Xanh)
    img_overlay = img.copy()
    if gt_coord:
        cv2.circle(img_overlay, gt_coord, 10, (0, 255, 0), 2)
    if pred_coord:
        cv2.circle(img_overlay, pred_coord, 6, (255, 0, 0), -1)
    dist_str = f"Sai số: {dist:.1f}px" if dist is not None else "Không phát hiện"
    axes[idx, 2].imshow(img_overlay)
    axes[idx, 2].set_title(f"Dự đoán TrackNetV4 (Đỏ) vs GT (Xanh) | {dist_str}", fontsize=11, fontweight='bold')
    axes[idx, 2].axis('off')

plt.tight_layout()
bounce_path = "reports/tracknet_v4_reports/tracknet_v4_bounce_evaluation.png"
plt.savefig(bounce_path, dpi=300)
plt.show()
print(f"💾 Đã lưu hình ảnh trực quan hóa tình huống bóng nảy vào: {bounce_path}")


## 9. Hướng Dẫn Tích Hợp TrackNetV4 Thay Thế YOLO vào Pipeline Dự Án
Sau khi hoàn thành huấn luyện, mô hình được lưu tại `models/tracknet_v4_ball_detector_best.keras`.

Để tích hợp vào `trackers/ball_tracker.py` thay cho YOLO, ta chỉ cần duy trì sliding window gồm 3 frame liên tiếp $(I_{t-1}, I_t, I_{t+1})$:

```python
import keras
import cv2
import numpy as np

class TrackNetV4BallTracker:
    def __init__(self, model_path="models/tracknet_v4_ball_detector_best.keras", conf_thresh=0.35):
        self.model = keras.models.load_model(model_path, compile=False)
        self.conf_thresh = conf_thresh
        self.target_size = (288, 512)

    def detect_frames(self, frames):
        """Dự đoán tọa độ bóng cho toàn bộ video bằng cụm 3 frame."""
        ball_detections = []
        num_frames = len(frames)
        
        for i in range(num_frames):
            # Lấy 3 frame: biên đầu lặp frame 0, biên cuối lặp frame cuối
            idx_p = max(0, i - 1)
            idx_c = i
            idx_n = min(num_frames - 1, i + 1)
            
            p_img = cv2.resize(frames[idx_p], (self.target_size[1], self.target_size[0])) / 255.0
            c_img = cv2.resize(frames[idx_c], (self.target_size[1], self.target_size[0])) / 255.0
            n_img = cv2.resize(frames[idx_n], (self.target_size[1], self.target_size[0])) / 255.0
            
            inp = np.expand_dims(np.concatenate([p_img, c_img, n_img], axis=-1), 0)
            preds = self.model(inp, training=False)
            heatmap = np.array(preds['ball_heatmap'][0, :, :, 0])
            
            if heatmap.max() >= self.conf_thresh:
                py, px = np.unravel_index(np.argmax(heatmap), heatmap.shape)
                # Map lại kích thước gốc của video
                orig_h, orig_w = frames[i].shape[:2]
                rx = (px / self.target_size[1]) * orig_w
                ry = (py / self.target_size[0]) * orig_h
                ball_detections.append({1: [rx - 8, ry - 8, rx + 8, ry + 8]})
            else:
                ball_detections.append({})
                
        return ball_detections
```
🎉 **Hoàn tất pipeline phát hiện bóng với độ chính xác cao bằng TrackNetV4!**
